# Data Loading and library imports

### Importing libraries

In [1]:
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
import seaborn as sns
import cv2
from sklearn.model_selection import train_test_split
from tensorflow import keras

I0000 00:00:1794415805.646029    8509 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794415806.363267    8509 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794415809.475798    8509 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


### Loading dataset

In [3]:
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "cracked & Non Cracked").is_dir()
)

Dataset_Path = PROJECT_ROOT / "cracked & Non Cracked"

Data = []

for image in Dataset_Path.rglob("*"):
    if image.suffix.lower() == ".jpg":
        with Image.open(image) as img:
            Data.append({
                "image_path": str(image),
                "label": image.parent.name
            })

cracked_dataset = pd.DataFrame(Data)

cracked_dataset[["image_path", "label"]].to_parquet(
    PROJECT_ROOT / "dataset.parquet",
    engine="pyarrow",
    index=False
)

In [ ]:
cracked_dataset["image_path"] = cracked_dataset["image_path"].apply(
    lambda p: str(Path(p).relative_to(Dataset_Path.parent))
)


In [9]:
cracked_dataset.to_parquet(
    PROJECT_ROOT / "dataset.parquet",
    engine="pyarrow",
    index=False
)

In [11]:
train_val, test = train_test_split(
    cracked_dataset,
    test_size=0.20,
    random_state=42,
    stratify=cracked_dataset["label"]
)

train, val = train_test_split(
    train_val,
    test_size=0.1875,
    random_state=42,
    stratify=train_val["label"]
)

train.to_parquet(PROJECT_ROOT / "train.parquet", index=False)
val.to_parquet(PROJECT_ROOT / "validation.parquet", index=False)
test.to_parquet(PROJECT_ROOT / "test.parquet", index=False)

In [12]:
train = pd.read_parquet(PROJECT_ROOT / "train.parquet")
val = pd.read_parquet(PROJECT_ROOT / "validation.parquet")
test = pd.read_parquet(PROJECT_ROOT / "test.parquet")

In [13]:
X_train = train["image_path"]
y_train = train["label"]

X_val = val["image_path"]
y_val = val["label"]

X_test = test["image_path"]
y_test = test["label"]

# Preprocessing

In [ ]:
train.head()